In [1]:
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


In [5]:
import os
os.getcwd()

'd:\\уник\\мага\\3сем\\МОХ\\IIS-LR1\\research'

In [9]:
df = pd.read_pickle(r"../heart_clean.pkl")  
df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### 3.Разбиение на тестовую и обучающую выборки. 
### 4.Создание переменных, содержащих названия столбцов с числовыми и категориальными признаками


In [10]:
# Целевая переменная
target_col = 'target'

# Числовые признаки
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# Категориальные признаки
car_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

# Разделяем выборку
X = df[num_cols + car_cols]
y = df[target_col]

# Стратифицированное разбиение 75% / 25%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((226, 13), (76, 13))

### 5,6.Создание pipeline

In [11]:
# Преобразования для числовых признаков
num_transformer = StandardScaler()

# Преобразования для категориальных признаков
cat_transformer = TargetEncoder()

# ColumnTransformer объединяет оба набора
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, car_cols)
    ]
)

# Используем RandomForest в качестве baseline модели
rf_clf = RandomForestClassifier(random_state=42)

# Полный pipeline: сначала препроцессинг, затем классификатор
baseline_pipe = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', rf_clf)
])

baseline_pipe

,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
# Обучение
baseline_pipe.fit(X_train, y_train)

# Получение метрик
y_pred = baseline_pipe.predict(X_test)
y_pred_proba = baseline_pipe.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")

Precision: 0.745
Recall:    0.854
F1-score:  0.795
ROC-AUC:   0.862


### 7,8. Mlflow

In [14]:
import mlflow
mlflow.set_tracking_uri("sqlite:///../mlflow/mlruns.db")

# Создание нового эксперимента с заданным артефактным хранилищем
mlflow.set_experiment("HeartDisease-Baseline")

2025/10/15 20:06:40 INFO mlflow.tracking.fluent: Experiment with name 'HeartDisease-Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///d:/уник/мага/3сем/МОХ/IIS-LR1/research/mlruns/1', creation_time=1760548000541, experiment_id='1', last_update_time=1760548000541, lifecycle_stage='active', name='HeartDisease-Baseline', tags={}>

### 9. Логирование

In [15]:
from mlflow.models import infer_signature

# Пример входных данных и сигнатура модели
input_example = X_train.head(5)
signature = infer_signature(X_train, baseline_pipe.predict(X_train.head(5)))

with mlflow.start_run(run_name="Baseline RandomForest"):

    # --- параметры модели ---
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", rf_clf.n_estimators)
    mlflow.log_param("random_state", rf_clf.random_state)

    # --- метрики качества ---
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # --- логирование модели ---
    mlflow.sklearn.log_model(
        sk_model=baseline_pipe,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="HeartDiseaseModel"
    )

d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'HeartDiseaseModel'.
Created version '1' of model 'HeartDiseaseModel'.
d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ip

### 10.

In [18]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, KBinsDiscretizer
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from mlflow.models import infer_signature
import numpy as np




poly_features = ['age', 'chol']
kbins_features = ['thalach']

poly = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler())
])

kbins = Pipeline([
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
    ('scale', StandardScaler())
])

extended_transformer = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('poly', poly, poly_features),
    ('kbins', kbins, kbins_features),
    ('cat', TargetEncoder(), car_cols)
])

rf_extended = RandomForestClassifier(random_state=42)
pipe_extended = Pipeline([
    ('transform', extended_transformer),
    ('clf', rf_extended)
])

pipe_extended.fit(X_train, y_train)

y_pred = pipe_extended.predict(X_test)
y_proba = pipe_extended.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"SKLearn Features -> F1={f1:.3f}, ROC_AUC={roc_auc:.3f}")

X_train_fe_cols = np.array(pipe_extended.named_steps['transform'].get_feature_names_out())
pd.Series(X_train_fe_cols).to_csv("feature_names_sklearn.csv", index=False)

d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


SKLearn Features -> F1=0.805, ROC_AUC=0.868


In [21]:
signature = infer_signature(X_train, pipe_extended.predict(X_train))

mlflow.set_experiment("HeartDisease-Baseline")
with mlflow.start_run(run_name="Sklearn_FeatureEng"):
    mlflow.log_artifact("feature_names_sklearn.csv")
    mlflow.log_metrics({"precision": precision, "recall": recall, "f1": f1, "roc_auc": roc_auc})
    mlflow.sklearn.log_model(pipe_extended, "model", registered_model_name="HeartDiseaseModel",signature=signature,input_example=X_train.head(5))

d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'HeartDiseaseModel' already exists. Creating a new version of this model...
Created version '4' of model 'HeartDiseaseModel'.


12.

In [20]:

from mlxtend.feature_selection import SequentialFeatureSelector as SFS

sample_X = X_train.sample(frac=0.3, random_state=42)
sample_y = y_train.loc[sample_X.index]

sfs = SFS(RandomForestClassifier(random_state=42),
          k_features='best', forward=True, floating=False,
          scoring='f1', cv=5, n_jobs=-1)
sfs.fit(sample_X, sample_y)

selected_features = list(sfs.k_feature_names_)
pd.Series(selected_features).to_csv("selected_features_mlxtend.csv", index=False)
print(f"otobrano{len(selected_features)} priznakov")
rf_selected = RandomForestClassifier(random_state=42)
rf_selected.fit(X_train[selected_features], y_train)

y_pred=rf_selected.predict(X_test[selected_features])
y_proba=rf_selected.predict_proba(X_test[selected_features])[:,1]

precision=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
f1=f1_score(y_test,y_pred)
roc_auc=roc_auc_score(y_test,y_proba)



otobrano6 priznakov


In [22]:
signature = infer_signature(X_train, pipe_extended.predict(X_train))

with mlflow.start_run(run_name="FeatureSelection_MLXtend"):
    mlflow.log_artifact("selected_features_mlxtend.csv")
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(pipe_extended, "model", registered_model_name="HeartDiseaseModel", signature=signature,input_example=X_train.head(5))



d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'HeartDiseaseModel' already exists. Creating a new version of this model...
Created version '5' of model 'HeartDiseaseModel'.


In [25]:
# ПУНКТ 14. Настройка гиперпараметров с Optuna
# ===============================================
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)

    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, max_features=max_features, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='f1').mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

best_model = RandomForestClassifier(**study.best_params, random_state=42)
best_model.fit(X_train, y_train)


y_pred=best_model.predict(X_test)
y_proba=best_model.predict_proba(X_test)[:,1]

precision=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
f1=f1_score(y_test,y_pred)
roc_auc=roc_auc_score(y_test,y_proba)

signature = infer_signature(X_train, pipe_extended.predict(X_train))
with mlflow.start_run(run_name="Optuna_RF_Best"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(best_model, "model", registered_model_name="HeartDiseaseModel", signature=signature,input_example=X_train.head(5))

[I 2025-10-15 20:38:25,877] A new study created in memory with name: no-name-53c0864c-2d93-4e2c-9c56-ffc8e4946238
[I 2025-10-15 20:38:26,313] Trial 0 finished with value: 0.8502566752799311 and parameters: {'n_estimators': 51, 'max_depth': 9, 'max_features': 0.7937395551655624}. Best is trial 0 with value: 0.8502566752799311.
[I 2025-10-15 20:38:27,275] Trial 1 finished with value: 0.8364739177324678 and parameters: {'n_estimators': 146, 'max_depth': 11, 'max_features': 0.546084252074262}. Best is trial 0 with value: 0.8502566752799311.
[I 2025-10-15 20:38:28,151] Trial 2 finished with value: 0.8666467852221784 and parameters: {'n_estimators': 146, 'max_depth': 5, 'max_features': 0.14604849185525784}. Best is trial 2 with value: 0.8666467852221784.
[I 2025-10-15 20:38:28,697] Trial 3 finished with value: 0.8458831908831907 and parameters: {'n_estimators': 75, 'max_depth': 12, 'max_features': 0.6901873738918775}. Best is trial 2 with value: 0.8666467852221784.
[I 2025-10-15 20:38:29,836

16

In [27]:

# ПУНКТ 16. Финальная Production-модель

from mlflow.models import infer_signature
import shutil
import os

# --- 1. Обучаем лучшую модель на всей выборке ---
final_model = best_model.fit(X, y)
y_pred=best_model.predict(X)
y_proba=best_model.predict_proba(X)[:,1]

precision=precision_score(y,y_pred)
recall=recall_score(y,y_pred)
f1=f1_score(y,y_pred)
roc_auc=roc_auc_score(y,y_proba)




# --- 2. Пример входных данных и сигнатура ---
input_example = X.head(5)
signature = infer_signature(X, final_model.predict(X.head(5)))



# Если есть файл с признаками — используем его
feature_file = "feature_names_sklearn.csv"
if not os.path.exists(feature_file):
    pd.Series(X.columns).to_csv(feature_file, index=False)

# --- 4. Логирование финальной модели ---

with mlflow.start_run(run_name="Production_Model"):
    # Логируем артефакты
    mlflow.log_artifact("requirements.txt")
    mlflow.log_artifact(feature_file)
    

    # Логируем модель с сигнатурой и примером входных данных
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="HeartDiseaseModel"
    )

    # Ставим тег Production
    mlflow.set_tag("stage", "Production")

d:\уник\мага\3сем\МОХ\IIS-LR1\.venv\lib\site-packages\mlflow\types\utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'HeartDiseaseModel' already exists. Creating a new version of this model...
Created version '8' of model 'HeartDiseaseModel'.
